# Exercise 3: Resource gap analysis

*The Turkic World · version 0.1.0*

> Choose a language, domain and community use case. Using the audit fields in the resource section, identify three resources needed for that use case, search for named releases, and record any gaps as unverified until the search scope is documented. Prioritise work by user needs, permissions and feasibility.

Worked example: **named-entity recognition for Turkmen news**, for example to index local news archives by people, places and organisations.

**Setup.** `pip install "turkicnlp==0.5.0" requests pandas`. Cells that query websites need internet access; if a request fails, the cell records the failure instead of a number. An unavailable value is not zero.

In [ ]:
import datetime
import requests
import pandas as pd

ACCESSED = datetime.date.today().isoformat()        # record the access date with every observation
HEADERS = {"User-Agent": "turkic-nlp-book-exercises/0.1 (educational use)"}

def get_json(url, **params):
    """Return parsed JSON, or {'_error': ...} if the request fails (never a fake value)."""
    try:
        r = requests.get(url, params=params or None, headers=HEADERS, timeout=30)
        r.raise_for_status()
        return r.json()
    except Exception as exc:
        return {"_error": f"{type(exc).__name__}: {exc}"}

def get_text(url):
    try:
        r = requests.get(url, headers=HEADERS, timeout=30)
        r.raise_for_status()
        return r.text
    except Exception as exc:
        return None

def github_org_repos(org):
    """All public repositories of a GitHub organisation (paged listing; avoids the search API's low rate limit)."""
    repos, page = [], 1
    while True:
        d = get_json(f"https://api.github.com/orgs/{org}/repos", per_page=100, page=page)
        if not isinstance(d, list):
            return repos, d.get("_error")
        repos += d
        if len(d) < 100:
            return repos, None
        page += 1

In [ ]:
import turkicnlp

USE_CASE = {"language": "Turkmen", "code": "tuk", "domain": "news", "task": "named-entity recognition",
            "users": "archive and newsroom staff searching Turkmen news by people, places and organisations"}

audit = []   # one row per resource: the audit fields from The Turkic World

# Need 1 — annotated evaluation data. Candidate: WikiANN (automatically derived from Wikipedia links)
d = get_json("https://datasets-server.huggingface.co/size", dataset="unimelb-nlp/wikiann", config="tk")
sizes = ({s["split"]: s["num_rows"] for s in d["size"]["splits"]} if "size" in d else f"not retrieved ({d.get('_error')})")
audit.append({"need": "evaluation data", "resource": "WikiANN (config tk)", "url": "https://huggingface.co/datasets/unimelb-nlp/wikiann",
              "size": sizes, "annotation": "automatic, PER/ORG/LOC", "domain": "Wikipedia", "accessed": ACCESSED,
              "status": ("found; domain and annotation do not match news NER" if isinstance(sizes, dict)
                         else "request failed; check the dataset page manually")})

# Need 2 — a ready-made model. Ask the toolkit what it offers for Turkmen
procs = turkicnlp.list_processors(USE_CASE["code"])
audit.append({"need": "NER model", "resource": "TurkicNLP 0.5.0 trained NER", "url": "https://github.com/turkic-nlp/turkicnlp",
              "size": "-", "annotation": "-", "domain": "-", "accessed": ACCESSED,
              "status": "found" if "ner" in procs else "no trained NER processor (LLM backend only; unevaluated)"})

# Need 3 — human-annotated Turkmen news NER training data
audit.append({"need": "training data", "resource": "human-annotated Turkmen news NER", "url": "-",
              "size": "-", "annotation": "-", "domain": "news", "accessed": ACCESSED,
              "status": "not found in this search (searched: Hugging Face, ACL Anthology, Syntax and Universal Dependencies inventory)"})

pd.set_option("display.max_colwidth", 80)
pd.DataFrame(audit)

## Worked solution (checked 18 October 2026)

| Need | What the search found | Gap |
|---|---|---|
| Evaluation data | WikiANN `tk`: 100 / 100 / 100 sentences, automatically derived from Wikipedia links, three entity types | No human-annotated **news** test set was found. |
| Model | TurkicNLP 0.5.0 lists no trained NER processor for Turkmen; the LLM backend can label entities but has not been evaluated on Turkmen | No evaluated model |
| Training data | Not found in the searched sources | Unverified until the search is broadened |

**Priority order and justification.**

1. **A small, human-annotated Turkmen news test set (a few hundred sentences).** Without it no system, including an LLM or a transferred Kazakh/Turkish model, can be evaluated for the users' domain. It is feasible for a small team, and permission is easiest to obtain for a narrow, documented sample of news articles whose licence allows annotation and redistribution.
2. **An evaluated baseline.** Compare zero-shot transfer (a multilingual model fine-tuned on KazNERD or Turkish NER), the LLM backend and, later, a model fine-tuned on Turkmen data, all on the same test set.
3. **Training data.** Annotate more news only if the baseline is not good enough for the users; the test set shows where it fails.

**Permissions and validation.** Record each article's source and licence, have two native speakers annotate an overlapping sample and report agreement, and agree with the newsroom or archive what may be published. WikiANN's existence does not close the gap: its domain (Wikipedia) and automatic annotation do not match the use case.

## Your turn — extension

Change `USE_CASE` to your own language, domain and users, replace the three needs with those of your use case, and justify your priority order in a cell below.